# Import

In [1]:
import models.juanchitocnn
import models.visiontransformer
import models.efficientnetb0
import data_loader
import torch
from FGSM_attack import FGSMAttacker
print('Import Done')

C:\Users\Win\anaconda3\envs\cs7643-a4\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Import Done


# Select model

In [2]:
select_model = 'EfficientNet' # ['EfficientNet', 'VisionTransformer', 'JuanchitoCNN']

# Other parameters
train_test_split_ratio = 0.8

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Select data directory

In [3]:
#csv_path = '/home/hice1/jguardia7/scratch/datasets/alessandrasala79/ai-vs-human-generated-dataset/versions/4/train.csv'
#image_folder = '/home/hice1/jguardia7/scratch/datasets/alessandrasala79/ai-vs-human-generated-dataset/versions/4/train_data'

csv_path = 'data/train.csv'
image_folder = 'data/train_data'

# Tuning - Feel free to change values in list for loop

In [ ]:
best_acc = 0
print(select_model)
print(device)
for epochs in [5, 10, 15]:
    for learning_rate in [0.01, 0.001, 0.0001]:
        for batch_size in [16, 32, 64]:
            for weight_decay in [0, 0.001, 0.01]:
                for momentum in [0, 0.1, 0.9]:
                    for optimizer in ['SGD', 'AdamW']:

                        if select_model == 'EfficientNet':
                            project_model = models.efficientnetb0.ProjectEfficientNet(epochs=epochs, learning_rate=learning_rate, batch_size=batch_size, optimizer=optimizer, momentum=momentum, weight_decay=weight_decay)
                        elif select_model == 'VisionTransformer':
                            project_model = models.visiontransformer.ProjectVisionTransformer(epochs=epochs, learning_rate=learning_rate, batch_size=batch_size, optimizer=optimizer, momentum=momentum, weight_decay=weight_decay)
                        elif select_model == 'JuanchitoCNN':
                            project_model = models.juanchitocnn.ProjectJuanchitoCNN(epochs=epochs, learning_rate=learning_rate, batch_size=batch_size, optimizer=optimizer, momentum=momentum, weight_decay=weight_decay)
                        else:
                            print('No valid model selected')
                        
                        regular_train_loader, regular_test_loader = data_loader.data_to_train_test_dataloaders(csv_path=csv_path, 
                                                                                                           image_folder=image_folder, 
                                                                                                           image_size=(224, 224), 
                                                                                                           split_ratio=train_test_split_ratio, 
                                                                                                           train_batch_size=batch_size, 
                                                                                                           test_batch_size=batch_size)
                        
                        project_model.data_load(regular_train_loader, regular_test_loader)
                        project_model.train(track_loss=False)
                        correct, total, incorrect_preds = project_model.evaluate()
                        accuracy = 100 * correct / total
                        if accuracy > best_acc:
                            best_acc = accuracy
                            print('Best Accuracy so far')
                            
                        print(f'Test Accuracy: {accuracy:.2f}% from N={total}, Epochs: {epochs}, Learning Rate: {learning_rate}, Batch Size: {batch_size}, Weight Decay: {weight_decay}, Momentum: {momentum}, Optimizer: {optmizer}')

Epoch 1/5:  18%|█████████▎                                          | 719/3998 [01:46<04:57, 11.01batch/s, loss=0.2519]